<a href="https://colab.research.google.com/github/disCrybee/Machine-Learning/blob/main/%D0%9B%D0%A05_%D1%87%D0%B0%D1%81%D0%BE%D0%B2%D1%96_%D1%80%D1%8F%D0%B4%D0%B8_%D0%9F%D0%B5%D1%80%D0%B5%D1%80%D0%B5%D1%81%D1%82%D0%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота 5. Часові ряди
## Прогнозування кліматичних даних (Delhi)

**Датасет:** https://www.kaggle.com/datasets/yug201/daily-climate-time-series-data-delhi-india

**Мета:** EDA, ACF/PACF, моделі ARIMA, Prophet, LSTM, порівняння за MAE/RMSE/MAPE.

In [ ]:
# Імпорти
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# Завантаження з Kaggle
import kagglehub, os
path = kagglehub.dataset_download('yug201/daily-climate-time-series-data-delhi-india')
print(os.listdir(path))
# Знайдіть файл з погодою (зазвичай kaggel_weather... або подібний)
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(csv_files)
df = pd.read_csv(os.path.join(path, csv_files[0]))
df.head()

## Підготовка даних

In [ ]:
# Адаптуйте назву колонки дати під ваш файл (часто 'date', 'DATE', 'datetime')
date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
temp_col = [c for c in df.columns if 'temp' in c.lower() or 'meantemp' in c.lower()][0]

df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col)
df = df.set_index(date_col)

# Залишаємо температуру
ts = df[temp_col].astype(float)
ts = ts.dropna()
print(ts.head())
print('Розмір:', len(ts))
print('Пропуски:', ts.isnull().sum())

In [ ]:
print(ts.describe())
ts.info()

## EDA

In [ ]:
ts.plot(title='Температура у часі')
plt.ylabel('Температура')
plt.show()

In [ ]:
rolling = ts.rolling(window=30).mean()
ts.plot(label='Факт', alpha=0.5)
rolling.plot(label='Ковзне середнє (30 днів)', color='red')
plt.legend()
plt.title('Ковзне середнє')
plt.show()

In [ ]:
ts.hist(bins=30)
plt.title('Гістограма температури')
plt.show()

plt.figure()
sns.boxplot(x=ts)
plt.title('Boxplot температури')
plt.show()

## ACF / PACF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts.dropna(), lags=40, ax=axes[0])
plot_pacf(ts.dropna(), lags=40, ax=axes[1], method='ywm')
plt.tight_layout()
plt.show()

## Train / Test (часовий порядок)

In [ ]:
split = int(len(ts) * 0.8)
train = ts.iloc[:split]
test = ts.iloc[split:]
print('Train:', len(train), 'Test:', len(test))

## ARIMA

In [ ]:
# Простий підбір (можна змінити порядок)
model_arima = ARIMA(train, order=(2, 1, 2))
fitted_arima = model_arima.fit()
print(fitted_arima.summary())

In [ ]:
forecast_arima = fitted_arima.forecast(steps=len(test))
forecast_arima.index = test.index

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mae_a = mean_absolute_error(test, forecast_arima)
rmse_a = np.sqrt(mean_squared_error(test, forecast_arima))
mape_a = mape(test, forecast_arima)
print(f'ARIMA → MAE={mae_a:.3f}, RMSE={rmse_a:.3f}, MAPE={mape_a:.2f}%')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test.index, test, label='Факт')
plt.plot(forecast_arima.index, forecast_arima, label='ARIMA', color='red')
plt.legend()
plt.title('ARIMA: факт vs прогноз')
plt.show()

## Prophet

In [ ]:
# !pip install prophet -q
from prophet import Prophet

df_prophet = train.reset_index()
df_prophet.columns = ['ds', 'y']

m = Prophet()
m.fit(df_prophet)

future = m.make_future_dataframe(periods=len(test))
forecast_p = m.predict(future)

pred_p = forecast_p.set_index('ds')['yhat'].iloc[-len(test):]
pred_p = pred_p.reindex(test.index)

mae_p = mean_absolute_error(test, pred_p)
rmse_p = np.sqrt(mean_squared_error(test, pred_p))
mape_p = mape(test, pred_p)
print(f'Prophet → MAE={mae_p:.3f}, RMSE={rmse_p:.3f}, MAPE={mape_p:.2f}%')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test.index, test, label='Факт')
plt.plot(pred_p.index, pred_p, label='Prophet', color='green')
plt.legend()
plt.title('Prophet: факт vs прогноз')
plt.show()

m.plot_components(forecast_p)
plt.show()

## LSTM

In [ ]:
# !pip install tensorflow -q
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

scaler = MinMaxScaler()
train_sc = scaler.fit_transform(train.values.reshape(-1, 1))
test_sc = scaler.transform(test.values.reshape(-1, 1))

def create_sequences(data, lookback=30):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

lookback = 30
X_tr, y_tr = create_sequences(train_sc, lookback)
X_te, y_te = create_sequences(np.vstack([train_sc[-lookback:], test_sc]), lookback)

X_tr = X_tr.reshape((X_tr.shape[0], X_tr.shape[1], 1))
X_te = X_te.reshape((X_te.shape[0], X_te.shape[1], 1))

model_lstm = Sequential([
    LSTM(50, activation='relu', input_shape=(lookback, 1)),
    Dense(1)
])
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.summary()

es = EarlyStopping(patience=5, restore_best_weights=True)
model_lstm.fit(X_tr, y_tr, epochs=30, batch_size=32, validation_split=0.1, callbacks=[es], verbose=1)

In [ ]:
pred_lstm_sc = model_lstm.predict(X_te)
pred_lstm = scaler.inverse_transform(pred_lstm_sc).flatten()

# Вирівнюємо довжину
y_test_lstm = test.values[-len(pred_lstm):]

mae_l = mean_absolute_error(y_test_lstm, pred_lstm)
rmse_l = np.sqrt(mean_squared_error(y_test_lstm, pred_lstm))
mape_l = mape(y_test_lstm, pred_lstm)
print(f'LSTM → MAE={mae_l:.3f}, RMSE={rmse_l:.3f}, MAPE={mape_l:.2f}%')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test_lstm, label='Факт')
plt.plot(pred_lstm, label='LSTM', color='orange')
plt.legend()
plt.title('LSTM: факт vs прогноз')
plt.show()

## Порівняння моделей

In [ ]:
summary = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'LSTM'],
    'MAE': [mae_a, mae_p, mae_l],
    'RMSE': [rmse_a, rmse_p, rmse_l],
    'MAPE %': [mape_a, mape_p, mape_l]
}).sort_values('MAE')
print(summary)

## Висновок

Виконано EDA, ACF/PACF, побудовано ARIMA, Prophet і LSTM.  
Порівняння за MAE, RMSE, MAPE показує, яка модель краще працює на тестовому періоді.  
Зазвичай Prophet добре ловить сезонність, LSTM — складні нелінійності, ARIMA — простіші ряди.  
Оберіть модель з найменшими помилками.